# Day 4 — Experiment Tracking & Evaluation

---

Fine-tuning is **iterative**. You'll run 10, 20, 50 experiments before you're happy. Without tracking, you can't tell which hyperparameter combo won.

Today:

1. **Weights & Biases (W&B)** — the standard experiment-tracking tool
2. **Evaluation** — automatic (perplexity, exact match) + LLM-as-judge
3. Why BLEU / ROUGE are barely useful anymore


## 1. W&B in one paragraph

You install `wandb`, get a free API key, and add `report_to="wandb"` to your training config. Every hyperparameter, every loss value, every checkpoint gets logged to a beautiful web dashboard. You can compare runs side-by-side, group by config, and share the results with a URL.

For freshers: **use W&B for every fine-tuning experiment.** It costs nothing and the workflow benefits are enormous.


In [ ]:
!pip install wandb --quiet

In [ ]:
import os, wandb

# Set your key from https://wandb.ai/authorize
os.environ["WANDB_API_KEY"] = "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"  # replace or put in .env
wandb.login()


In [ ]:
# Enable W&B in your SFTConfig (Colab / GPU cell)
# from trl import SFTConfig
# config = SFTConfig(
#     output_dir="out",
#     report_to="wandb",                # <-- this line
#     run_name="triage-r16-lr2e-4",     # <-- searchable name
#     ...
# )


After training, open `https://wandb.ai/<you>/<project>` — you'll see:

- Loss curves (train + eval)
- All your hyperparameters as columns
- System stats (GPU mem, throughput)
- Every run in a table you can filter

**Naming convention worth adopting:** `<task>-r<rank>-lr<lr>-ep<epochs>`, e.g. `triage-r16-lr2e-4-ep3`. Future-you will thank present-you.


## 2. Evaluation — the honest kind

**The metric that actually matters: does the model do the right thing on inputs it hasn't seen?**

Three levels of eval, in the order you should use them:

### Level 1 — Manual spot-check
Grab 10-20 held-out examples. Run the model. Read each output. Score correct / partial / wrong.

**Do this before every "is my fine-tune better?" claim.** Nothing beats reading outputs with your own eyes.


In [ ]:
import json, urllib.request
from pathlib import Path

EVAL_PATH = Path("../Day_1_When_To_Fine_Tune/triage_eval.jsonl")
eval_examples = [json.loads(l) for l in EVAL_PATH.read_text().splitlines() if l.strip()]

def ollama_chat(model, messages, max_tokens=20):
    body = json.dumps({
        "model": model,
        "messages": messages,
        "stream": False,
        "options": {"temperature": 0, "num_predict": max_tokens},
    }).encode()
    req = urllib.request.Request(
        "http://localhost:11434/api/chat",
        data=body, headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(req) as r:
        return json.loads(r.read())["message"]["content"].strip()

def spot_check(model, examples, n=10):
    for row in examples[:n]:
        expected = row["messages"][-1]["content"].strip()
        got = ollama_chat(model, row["messages"][:-1])
        mark = "OK" if got.lower() == expected.lower() else "!!"
        print(f"{mark}  expected={expected!r:20} got={got!r}")

spot_check("llama3.1:8b", eval_examples)


### Level 2 — Exact match / regex accuracy
For classification and structured-output tasks, you can auto-score.


In [ ]:
def accuracy(model, examples):
    correct = 0
    for row in examples:
        expected = row["messages"][-1]["content"].strip().lower()
        got = ollama_chat(model, row["messages"][:-1]).lower()
        if got == expected:
            correct += 1
    return correct / len(examples)

print(f"llama3.1:8b accuracy: {accuracy('llama3.1:8b', eval_examples):.1%}")


### Level 3 — LLM-as-judge
When outputs are free-form (summaries, code, chat), have GPT-4 or Claude judge if the answer is *good enough*.

Prompt template:

```
You are a strict evaluator. Given a question, an expected answer, and a model answer,
return a JSON object {"score": 0-5, "reason": "..."}.

Question: <q>
Expected: <exp>
Model:    <got>
```

Use LLM-as-judge sparingly — it costs money per eval example — but it's the most flexible.


In [ ]:
JUDGE_SYSTEM = (
    "You are a strict evaluator. Given a question, an expected answer, and a model "
    'answer, return only a JSON object of the form {"score": 0-5, "reason": "..."}.'
)

def judge(judge_model, question, expected, got):
    user = f"Question: {question}\nExpected: {expected}\nModel:    {got}"
    raw = ollama_chat(
        judge_model,
        [{"role": "system", "content": JUDGE_SYSTEM},
         {"role": "user", "content": user}],
        max_tokens=120,
    )
    try:
        return json.loads(raw[raw.index("{"): raw.rindex("}") + 1])
    except (ValueError, json.JSONDecodeError):
        return {"score": None, "reason": raw}

for row in eval_examples:
    q = row["messages"][-2]["content"]
    expected = row["messages"][-1]["content"]
    got = ollama_chat("llama3.1:8b", row["messages"][:-1])
    verdict = judge("llama3.1:8b", q, expected, got)
    print(f"q={q!r}\n  got={got!r}  verdict={verdict}\n")


## 3. Why BLEU / ROUGE are barely used anymore

- **BLEU / ROUGE** measure n-gram overlap with a reference. They were the standard in the 2010s.
- Modern LLM outputs can be *correct but worded very differently* from the reference → BLEU says "bad" when it's fine.
- LLM-as-judge is now the standard for free-form eval.

**Know their names for interviews. Don't use them for RAG or general chat fine-tuning.** They're still fine for narrow tasks like translation.


# Swap in a second ollama model (e.g. your fine-tuned "triage" from Day 5) to compare.
BASE_MODEL = "llama3.1:8b"
FT_MODEL   = "llama3.1:8b"   # replace with your fine-tune once you have one

base_acc = accuracy(BASE_MODEL, eval_examples)
ft_acc   = accuracy(FT_MODEL,   eval_examples)
print(f"base: {base_acc:.1%}   fine-tuned: {ft_acc:.1%}   delta: {(ft_acc-base_acc)*100:+.1f} pp")


In [ ]:
# Pseudocode - swap in your loaders
# base_acc = accuracy(base_model, tokenizer, eval_examples)
# ft_acc   = accuracy(ft_model,   tokenizer, eval_examples)
# print(f"base: {base_acc:.1%}   fine-tuned: {ft_acc:.1%}   delta: +{(ft_acc-base_acc)*100:.1f} pp")


If your fine-tune isn't beating the base model on your eval set, **do not ship it**. Either:

- More/better data
- Different hyperparameters (rank, LR, epochs)
- Or accept that the base model was already good enough (this is common!)


## Recap

- Track every fine-tune in **W&B**. It's free and makes comparing experiments trivial.
- Always eval on a **held-out set**. Never on train.
- Start with **manual spot-checks**, then move to **exact match** or **LLM-as-judge**.
- BLEU/ROUGE mostly obsolete for modern LLM work. Know the names, don't use them.
- **Ship only if fine-tune > base on your eval set.**
- **Next class:** merging, exporting, and serving your fine-tuned model.
